# Notebook lineage prototype

This notebook explores whether a Fabric workload can extract lineage information from other notebooks. The idea is to keep the logic entirely inside the Fabric tenant and to analyze notebook source without sending raw customer data to any backend.

The prototype does three things:
- scans local notebook files in the workspace
- extracts references to lakehouses, warehouses, tables, and SQL objects
- infers likely transformation steps from notebook code patterns such as `select`, `join`, `groupBy`, `withColumn`, `union`, and `merge`

This is a practical proof of concept, not a guarantee that every notebook pattern can be parsed perfectly.

In [ ]:
import ast
import json
import re
from pathlib import Path
from collections import defaultdict

import pandas as pd

# Common helpers
def normalize_name(value):
    if value is None:
        return ""
    text = str(value).strip()
    return re.sub(r'[^A-Za-z0-9_\./-]+', '', text)

def safe_json_loads(value):
    if value is None:
        return {}
    if isinstance(value, dict):
        return value
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return {}
        try:
            return json.loads(text)
        except Exception:
            return {}
    return {}

def is_notebook_file(path):
    return path.name.endswith('.ipynb') and '2_LineageWorkbench_Notebook_Extraction' not in path.name

# Try to detect local notebook files in the current workspace. In Fabric notebooks,
# this may be a mounted folder or the current working directory.
candidate_roots = []
for root in [Path.cwd(), Path('/lakehouse/default'), Path('/workspace')]:
    if root.exists():
        candidate_roots.append(root)

notebook_files = []
for root in candidate_roots:
    for path in root.rglob('*.ipynb'):
        if is_notebook_file(path):
            notebook_files.append(path.resolve())

# Fallback: if nothing was found, inspect the current folder only.
if not notebook_files:
    notebook_files = sorted(Path.cwd().glob('*.ipynb'))

notebook_files = sorted(set(str(p) for p in notebook_files))
print(f'Found {len(notebook_files)} notebook files')
notebook_files[:10]

In [ ]:
def read_notebook(path):
    path = Path(path)
    try:
        content = json.loads(path.read_text(encoding='utf-8'))
    except Exception as ex:
        return {
            'path': str(path),
            'error': str(ex),
            'cells': []
        }

    cells = []
    for idx, cell in enumerate(content.get('cells', [])):
        source = ''.join(cell.get('source', []))
        cells.append({
            'index': idx,
            'type': cell.get('cell_type', 'code'),
            'source': source
        })
    return {
        'path': str(path),
        'cells': cells
    }

notebook_data = [read_notebook(p) for p in notebook_files]
notebook_data = [nb for nb in notebook_data if nb.get('cells')]
notebook_data[:2]

In [ ]:
def extract_sql_objects(text):
    patterns = [
        r'(?is)\bfrom\s+`?([A-Za-z0-9_./-]+)`?',
        r'(?is)\bjoin\s+`?([A-Za-z0-9_./-]+)`?',
        r'(?is)\binto\s+`?([A-Za-z0-9_./-]+)`?',
        r'(?is)\bmerge\s+into\s+`?([A-Za-z0-9_./-]+)`?',
        r'(?is)\bupdate\s+`?([A-Za-z0-9_./-]+)`?',
        r'(?is)\bcreate\s+table\s+`?([A-Za-z0-9_./-]+)`?'
    ]
    refs = set()
    for pattern in patterns:
        refs.update(re.findall(pattern, text))
    return sorted(refs)

def extract_python_objects(text):
    refs = set()
    for pattern in [
        r'\b(?:spark\.|sql\.|read\.|table\()\s*.*?([A-Za-z0-9_./-]{3,})',
        r"\b(?:Lakehouse|Warehouse|lakehouse|warehouse)\b[^\n]{0,40}['\"]([A-Za-z0-9_./-]+)['\"]",
        r'([A-Za-z0-9_./-]+\.[A-Za-z0-9_./-]+)'
    ]:
        refs.update(re.findall(pattern, text))
    return sorted({normalize_name(r) for r in refs if normalize_name(r)})

def infer_transformations(text):
    text_lower = text.lower()
    transforms = []
    for signal, label in [
        ('withcolumn', 'withColumn'),
        ('selectexpr', 'selectExpr'),
        ('select(', 'select'),
        ('join(', 'join'),
        ('groupby(', 'groupBy'),
        ('agg(', 'agg'),
        ('union(', 'union'),
        ('merge', 'merge'),
        ('write.mode', 'write'),
        ('saveastable', 'saveAsTable')
    ]:
        if signal in text_lower:
            transforms.append(label)
    return sorted(set(transforms))

def extract_notebook_references(nb):
    refs = []
    for cell in nb.get('cells', []):
        source = cell.get('source', '')
        if not source.strip():
            continue
        sql_refs = extract_sql_objects(source)
        py_refs = extract_python_objects(source)
        transform_refs = infer_transformations(source)
        if sql_refs or py_refs or transform_refs:
            refs.append({
                'cell_index': cell.get('index'),
                'cell_type': cell.get('type'),
                'sql_refs': sql_refs,
                'python_refs': py_refs,
                'transformations': transform_refs
            })
    return refs

all_notebook_refs = []
for nb in notebook_data:
    all_notebook_refs.append({
        'path': nb.get('path'),
        'references': extract_notebook_references(nb)
    })

all_notebook_refs[:2]

In [ ]:
def build_lineage_rows():
    rows = []
    for nb in all_notebook_refs:
        notebook_path = nb.get('path', '')
        for ref in nb.get('references', []):
            cell_index = ref.get('cell_index')
            sql_refs = ref.get('sql_refs', [])
            py_refs = ref.get('python_refs', [])
            transformations = ref.get('transformations', [])
            candidates = sql_refs + py_refs
            if not candidates and not transformations:
                continue

            rows.append({
                'notebook': notebook_path.split('/')[-1],
                'path': notebook_path,
                'cell_index': cell_index,
                'object_refs': sorted(set(candidates)),
                'transformations': transformations,
                'source_type': 'notebook'
            })
    return rows

lineage_df = pd.DataFrame(build_lineage_rows())
lineage_df

In [ ]:
# Prepare a compact lineage summary that can be handed to a front-end or saved to a Delta table.
summary_df = lineage_df.copy()
summary_df['object_refs'] = summary_df['object_refs'].apply(lambda items: ', '.join(items) if isinstance(items, list) else str(items))
summary_df['transformations'] = summary_df['transformations'].apply(lambda items: ', '.join(items) if isinstance(items, list) else str(items))
summary_df = summary_df[['notebook', 'cell_index', 'object_refs', 'transformations', 'source_type']]
summary_df

## What this prototype extracts

This prototype is designed to answer a practical question: can a notebook infer notebook-to-dataset lineage early enough to help a workload build a lineage graph?

It does this by looking for patterns such as:
- SQL `FROM`, `JOIN`, `MERGE INTO`, `UPDATE`, and `INSERT INTO` clauses
- Python `spark.table()`, `spark.read...load()`, `write.mode`, and `saveAsTable()` patterns
- transformation operators such as `select`, `join`, `groupBy`, `agg`, `withColumn`, and `union`

This is not a full semantic parser, but it is strong enough to create a useful first-pass lineage map from notebook code and to surface likely lakehouse/warehouse dependencies.

In a production design, the same approach can be extended with:
- workspace metadata lookup for notebook item IDs
- Lakehouse/Warehouse inventory lookup
- notebook code normalization
- a confidence score for inferred transformations
- a final graph export to Delta tables or JSON for the frontend UI